# RevuLens Pilot Experiment (Colab-Ready Notebook)
## Cumulative Pipeline: Preprocessing, Baseline, Feature Extraction & LinearSVC Model Selection

Thesis Pilot comparing frozen encoders against TF-IDF SVM baseline for Genuine (0) vs Computer-Generated (1) review classification.

In [ ]:
# --- Configuration & Paths ---
# Change these paths if running in Google Colab (e.g. '/content/fake reviews dataset.csv')
SALMINEN_DATA_PATH = r'fake reviews dataset.csv'  # update to local or colab path
RANDOM_SEED = 42
RESULTS_DIR = 'results'
DATA_CACHE_DIR = 'data'

CANDIDATE_ENCODERS = [
    'distilbert-base-multilingual-cased',
    'xlm-roberta-base',
    'jcblaise/roberta-tagalog-base'
]

import os
import re
import html
import json
import random
import time
import tracemalloc
from typing import Dict, Any, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    brier_score_loss, confusion_matrix
)

def set_seed(seed=RANDOM_SEED):
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        else:
            torch.set_num_threads(min(4, os.cpu_count() or 1))
    except ImportError:
        pass

set_seed(RANDOM_SEED)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(DATA_CACHE_DIR, exist_ok=True)
print('Setup complete. Seed set to', RANDOM_SEED)

In [ ]:
# --- Text Normalization ---
def normalize_text(text: Any) -> str:
    if not isinstance(text, str):
        return ''
    text = html.unescape(text)
    text = re.sub(r'<[^>]+>', ' ', text)  # remove HTML tags
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)  # remove URLs
    text = text.lower()  # lowercase
    text = re.sub(r'([!?,.:;])\1+', r'\1', text)  # normalize repeated punctuation
    text = re.sub(r'\s+', ' ', text).strip()  # normalize whitespace
    return text

print('Normalized sample:', normalize_text('<div>Check https://example.com!!! BEST product ever!</div>'))

In [ ]:
# --- Dataset Loader (Supports Salminen and Future Schema) ---
def load_dataset(filepath: str, verbose: bool = True) -> pd.DataFrame:
    if not os.path.exists(filepath):
        raise FileNotFoundError(f'File not found: {filepath}')
    df = pd.read_csv(filepath)
    
    text_col = 'text_' if 'text_' in df.columns else ('text' if 'text' in df.columns else 'review')
    df['clean_text'] = df[text_col].apply(normalize_text)
    
    label_map = {
        'OR': 0, 'or': 0, 'Original': 0, 'genuine': 0, 'Genuine': 0, 0: 0, 0.0: 0, '0': 0,
        'CG': 1, 'cg': 1, 'Computer-Generated': 1, 'fake': 1, 'Fake': 1, 1: 1, 1.0: 1, '1': 1
    }
    df['label'] = df['label'].map(label_map).astype(int)
    if 'language' not in df.columns:
        df['language'] = 'English'
        
    # Drop empty and duplicates
    df = df[df['clean_text'].str.strip() != '']
    df = df.drop_duplicates(subset=['clean_text'], keep='first')
    df['text'] = df['clean_text']
    
    if verbose:
        print(f'Loaded {len(df)} valid records. Labels:\n{df["label"].value_counts()}')
    return df[['text', 'label', 'language']].reset_index(drop=True)

In [ ]:
# --- Stratified 80/10/10 Split ---
def split_dataset(df: pd.DataFrame, seed: int = RANDOM_SEED, save_path: Optional[str] = None):
    if save_path and os.path.exists(save_path):
        with open(save_path, 'r', encoding='utf-8') as f:
            return json.load(f)
    indices = np.arange(len(df))
    labels = df['label'].values
    train_idx, temp_idx = train_test_split(indices, test_size=0.20, random_state=seed, stratify=labels)
    val_idx, test_idx = train_test_split(temp_idx, test_size=0.50, random_state=seed, stratify=labels[temp_idx])
    splits = {'train': train_idx.tolist(), 'val': val_idx.tolist(), 'test': test_idx.tolist()}
    if save_path:
        os.makedirs(os.path.dirname(os.path.abspath(save_path)), exist_ok=True)
        with open(save_path, 'w', encoding='utf-8') as f:
            json.dump(splits, f, indent=2)
    return splits

In [ ]:
# --- Calibration & Evaluation Metrics ---
class ECEEvaluator:
    @staticmethod
    def compute(y_true: np.ndarray, prob_pos: np.ndarray, n_bins: int = 10) -> float:
        bin_boundaries = np.linspace(0.0, 1.0, n_bins + 1)
        ece = 0.0
        n = len(y_true)
        for i in range(n_bins):
            b_low, b_high = bin_boundaries[i], bin_boundaries[i+1]
            in_bin = (prob_pos >= b_low) & (prob_pos <= b_high if i == n_bins - 1 else prob_pos < b_high)
            count = np.sum(in_bin)
            if count > 0:
                acc_bin = np.mean(y_true[in_bin])
                conf_bin = np.mean(prob_pos[in_bin])
                ece += (count / n) * np.abs(acc_bin - conf_bin)
        return float(ece)

def compute_metrics(y_true: np.ndarray, prob_cg: np.ndarray, threshold: float = 0.50):
    y_pred = (prob_cg >= threshold).astype(int)
    confidences = np.maximum(prob_cg, 1.0 - prob_cg)
    acc = accuracy_score(y_true, y_pred)
    prec_cg = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
    rec_cg = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    f1_cg = f1_score(y_true, y_pred, pos_label=1, zero_division=0)
    brier = brier_score_loss(y_true, prob_cg)
    ece = ECEEvaluator.compute(y_true, prob_cg, n_bins=10)
    corr_mask = (y_pred == y_true)
    incorr_mask = (y_pred != y_true)
    return {
        'accuracy': float(acc),
        'precision_cg': float(prec_cg),
        'recall_cg': float(rec_cg),
        'f1_cg': float(f1_cg),
        'brier_score': float(brier),
        'ece': float(ece),
        'mean_conf_correct': float(np.mean(confidences[corr_mask])) if np.any(corr_mask) else 0.0,
        'mean_conf_incorrect': float(np.mean(confidences[incorr_mask])) if np.any(incorr_mask) else 0.0,
        'confusion_matrix': confusion_matrix(y_true, y_pred).tolist()
    }

In [ ]:
# --- TF-IDF Baseline Training & Calibration ---
def train_baseline_tfidf(train_texts, train_labels, val_texts, val_labels, c_grid=None):
    if c_grid is None:
        c_grid = [0.01, 0.1, 0.5, 1.0, 5.0, 10.0]
    vec = TfidfVectorizer(ngram_range=(1, 2), max_features=25000, sublinear_tf=True)
    X_tr = vec.fit_transform(train_texts)
    X_va = vec.transform(val_texts)
    
    scaler = StandardScaler(with_mean=False)
    X_tr_sc = scaler.fit_transform(X_tr)
    X_va_sc = scaler.transform(X_va)
    
    best_c, best_f1 = c_grid[0], -1.0
    for c in c_grid:
        svc = LinearSVC(C=c, random_state=RANDOM_SEED, max_iter=2000, dual=False)
        svc.fit(X_tr_sc, train_labels)
        f1 = f1_score(val_labels, svc.predict(X_va_sc), pos_label=1, zero_division=0)
        if f1 > best_f1:
            best_f1, best_c = f1, c
            
    print(f'Selected optimal C: {best_c} (Validation F1={best_f1:.4f})')
    base_svc = LinearSVC(C=best_c, random_state=RANDOM_SEED, max_iter=2000, dual=False)
    calibrated = CalibratedClassifierCV(estimator=base_svc, method='sigmoid', cv=5)
    calibrated.fit(X_tr_sc, train_labels)
    val_probas = calibrated.predict_proba(X_va_sc)[:, 1]
    metrics = compute_metrics(val_labels, val_probas)
    return calibrated, scaler, vec, metrics, val_probas

In [ ]:
# --- Frozen Encoders & Mean Pooling Feature Extraction ---
import torch
from transformers import AutoTokenizer, AutoModel

def mean_pooling(token_embeddings, attention_mask):
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, dim=1)
    sum_mask = torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)
    return sum_embeddings / sum_mask

class FeatureExtractor:
    def __init__(self, model_name: str, device: Optional[str] = None, batch_size: int = 128):
        self.model_name = model_name
        self.batch_size = batch_size
        self.device = torch.device(device) if device else torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(self.device)
        self.model.eval()
        for param in self.model.parameters():
            param.requires_grad = False
            
    def extract(self, texts: List[str], max_length: int = 128) -> np.ndarray:
        all_embs = []
        for i in range(0, len(texts), self.batch_size):
            batch = texts[i:i + self.batch_size]
            enc = self.tokenizer(batch, padding=True, truncation=True, max_length=max_length, return_tensors='pt').to(self.device)
            with torch.inference_mode():
                outputs = self.model(**enc)
                pooled = mean_pooling(outputs[0], enc['attention_mask'])
                all_embs.append(pooled.cpu().numpy().astype(np.float32))
        return np.vstack(all_embs) if all_embs else np.empty((0, 768), dtype=np.float32)

def get_cache_filename(model_name: str, split_name: str, max_length: int = 128) -> str:
    safe_name = model_name.replace('/', '_').replace('-', '_')
    return f'embeddings_{safe_name}_{split_name}_len{max_length}.npy'

def get_or_compute_embeddings(model_name, split_name, texts, cache_dir=DATA_CACHE_DIR, max_length=128, batch_size=128, extractor=None):
    os.makedirs(cache_dir, exist_ok=True)
    path = os.path.join(cache_dir, get_cache_filename(model_name, split_name, max_length))
    if os.path.exists(path):
        print(f'Loading cached embeddings: {path}')
        return np.load(path)
    if extractor is None:
        extractor = FeatureExtractor(model_name, batch_size=batch_size)
    embs = extractor.extract(texts, max_length=max_length)
    np.save(path, embs)
    print(f'Saved {len(texts)} embeddings -> {path}')
    return embs

In [ ]:
# --- LinearSVC Training, Calibration & Validation Evaluation ---
def train_linear_svc_pipeline(X_train, y_train, X_val, y_val, c_grid=None, seed=RANDOM_SEED):
    if c_grid is None:
        c_grid = [0.01, 0.1, 0.5, 1.0, 5.0, 10.0]
    scaler = StandardScaler(with_mean=True)
    X_tr_sc = scaler.fit_transform(X_train)
    X_va_sc = scaler.transform(X_val)
    
    best_c, best_f1 = c_grid[0], -1.0
    for c in c_grid:
        svc = LinearSVC(C=c, random_state=seed, max_iter=2000, dual=False)
        svc.fit(X_tr_sc, y_train)
        f1 = f1_score(y_val, svc.predict(X_va_sc), pos_label=1, zero_division=0)
        if f1 > best_f1:
            best_f1, best_c = f1, c
            
    base_svc = LinearSVC(C=best_c, random_state=seed, max_iter=2000, dual=False)
    calibrated = CalibratedClassifierCV(estimator=base_svc, method='sigmoid', cv=5)
    calibrated.fit(X_tr_sc, y_train)
    val_probas = calibrated.predict_proba(X_va_sc)[:, 1]
    metrics = compute_metrics(y_val, val_probas)
    return calibrated, scaler, best_c, metrics, val_probas

def select_best_pipeline(results, speed_lookup=None):
    sorted_m = sorted(results.keys(), key=lambda m: results[m]['metrics']['f1_cg'], reverse=True)
    top_m = sorted_m[0]
    top_f1 = results[top_m]['metrics']['f1_cg']
    within_1pt = [m for m in sorted_m if (top_f1 - results[m]['metrics']['f1_cg']) <= 0.01]
    if len(within_1pt) == 1 or speed_lookup is None:
        return top_m, f'Highest F1={top_f1:.4f}'
    fastest = max(within_1pt, key=lambda m: speed_lookup.get(m, 0.0))
    return fastest, f'Selected {fastest} (faster/smaller within 1pt of {top_m})'

def evaluate_seed_stability(X_train, y_train, X_val, y_val, best_c, seeds=[42, 7, 123]):
    f1_list = []
    for s in seeds:
        scaler = StandardScaler(with_mean=True)
        X_tr = scaler.fit_transform(X_train)
        X_va = scaler.transform(X_val)
        base_svc = LinearSVC(C=best_c, random_state=s, max_iter=2000, dual=False)
        calibrated = CalibratedClassifierCV(estimator=base_svc, method='sigmoid', cv=5)
        calibrated.fit(X_tr, y_train)
        probs = calibrated.predict_proba(X_va)[:, 1]
        f1_list.append(compute_metrics(y_val, probs)['f1_cg'])
    return {'seeds': seeds, 'f1_scores': f1_list, 'mean_f1': float(np.mean(f1_list)), 'std_f1': float(np.std(f1_list))}

def plot_confusion_matrices(pipeline_results, output_path='results/confusion_matrices.png'):
    n = len(pipeline_results)
    fig, axes = plt.subplots((n + 1) // 2, 2, figsize=(10, 4 * ((n + 1) // 2)))
    axes = np.array([axes]).flatten()
    for idx, (name, res) in enumerate(pipeline_results.items()):
        cm = np.array(res['metrics']['confusion_matrix'])
        ax = axes[idx]
        ax.imshow(cm, cmap='Blues')
        ax.set_title(f"{name.split('/')[-1]}\nF1={res['metrics']['f1_cg']:.4f}")
        for i in range(2):
            for j in range(2):
                ax.text(j, i, f'{cm[i, j]:,}', ha='center', va='center', color='white' if cm[i, j] > cm.max()/2 else 'black')
    plt.tight_layout()
    plt.savefig(output_path, dpi=200)
    plt.close()
    return output_path